# DKI — hyperparameter sweep for the nonlinear keystoneness model

Companion to [`dki_keystoneness_workflow.ipynb`](dki_keystoneness_workflow.ipynb).
That notebook trains **one** nonlinear replicator model with a hand-picked
config (`lr=1e-3`, `min_lr=1e-5`, `batch_size=500`, `hidden_mult=2`). This
notebook **searches** that space so we can be confident we are shipping the
best model before we trust its keystoneness numbers.

## What we sweep and why
The model-selection metric is **validation Bray-Curtis** (`result.best_val_loss`)
- the exact quantity `train()` already early-stops on. Lower is better. We sweep:

| dimension | why it matters for the nonlinear field |
|---|---|
| **`lr`** | the high-dim replicator field diverges at `1e-2`; the sweet spot is data-dependent |
| **`min_lr`** | the cosine floor; too high keeps the model jittering, too low freezes it early |
| **`batch_size`** | gradient variance vs coverage (auto-capped to `n_train`) |
| **`hidden_mult`** | width of the SiLU fitness `fc2(SiLU(fc1(y)))` - model capacity |
| **`weight_decay`** | L2 regularisation against overfitting the assembly rules |
| **`grad_clip`** | tames stiff-field gradient spikes (fewer skipped steps) |
| **`loss` / `alpha`** | `bc` vs `composite` (`a.BC + (1-a).CLR-MSE`) for rare-species accuracy |
| **`consistency_weight`** | self-consistency regulariser -> tighter fixed points |

`nonlinear=True` is **fixed** - this notebook is specifically about nailing the
nonlinear model (the linear collapse is studied in
`dki_linear_vs_nonlinear.ipynb`).

## How the search runs
1. **Random search** over the space (default, `N_TRIALS` draws) - far more
   sample-efficient than a full grid at this dimensionality. A grid mode is
   provided too.
2. Each trial trains with a **reduced epoch budget** + early stopping for speed.
3. Diverged trials (non-finite loss -> skipped optimiser steps) are flagged, not
   silently ranked as "good".
4. We rank by val BC, visualise per-dimension marginals + a couple of 2-D
   interactions, optionally re-check the top configs across seeds, then
   **retrain the winner at the full budget** and save it for the keystoneness
   workflow.


## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/metagenAu/DKI.git'
BRANCH   = 'claude/keen-turing-E4Z8Z'   # this notebook's branch; change to 'main' once merged
REPO_DIR = '/content/DKI'

# On Colab this clones the repo; locally, just run from the repo root.
if os.path.isdir('/content') and not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
sys.path.insert(0, os.getcwd())

import itertools, json, time, random
from dataclasses import replace

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dki.data import load_dataset
from dki.train import TrainConfig, train
from dki.infer import predict
from dki.losses import bray_curtis
from dki.device import auto_device

print('torch', torch.__version__, '| device', auto_device())

## 2. Data

Loads the data **once** and reuses the same object for every trial, so all
configs see the identical train/val split - the sweep then measures the effect
of the hyperparameters, not of a shifting split. Point `DATA_DIR` at any folder
with a `Ptrain.csv` (and optionally `Ptest.csv` / `Ztest.csv`).

`MIN_READS` is the same read-depth QC filter as the main notebook (0 = off);
leave it at 0 for data already in relative abundance or for the bundled
ground-truth set.

In [ ]:
DATA_DIR     = os.path.join(os.getcwd(), 'data')
SEED         = 0       # fixed split + init seed for the whole search (fair comparison)
VAL_FRACTION = 0.2
MIN_READS    = 0.0     # read-depth QC on raw Ptrain (0 = off)

data = load_dataset(DATA_DIR, val_fraction=VAL_FRACTION, seed=SEED, min_reads=MIN_READS)
N_TRAIN = data.z_train.shape[0]
print(f'n_species={data.n_species}  train={N_TRAIN}  val={data.z_val.shape[0]}  '
      f"test={0 if data.z_test is None else data.z_test.shape[0]}")
HAS_TEST = data.p_test is not None and data.z_test is not None

## 3. Define the search space

Edit the dictionaries below to widen / narrow the search. Values are sampled
**independently** per trial in random mode, or crossed into a full grid in grid
mode. `batch_size` is auto-capped to `N_TRAIN` inside the trial runner, so
oversized values simply fall back to the full batch.

`SWEEP_EPOCHS` is deliberately smaller than the 400 the main notebook uses -
the sweep just needs to rank configs, and early stopping trims most runs well
before the cap. Bump it if your val curves clearly have not plateaued.

In [ ]:
# --- search space ---------------------------------------------------------
SPACE = {
    'lr':                 [3e-4, 1e-3, 3e-3, 5e-3],
    'min_lr':             [1e-6, 1e-5, 1e-4],
    'batch_size':         [50, 100, 250, 500],     # capped to N_TRAIN
    'hidden_mult':        [1, 2, 4],
    'weight_decay':       [0.0, 1e-5, 1e-4],
    'grad_clip':          [0.5, 1.0, 2.0],
    'loss':               ['bc', 'composite'],
    'alpha':              [0.3, 0.5],               # only used when loss == 'composite'
    'consistency_weight': [0.0, 0.1],
}

# --- search controls ------------------------------------------------------
SEARCH       = 'random'   # 'random' or 'grid'
N_TRIALS     = 30         # random-search budget (ignored for grid)
SWEEP_EPOCHS = 250        # reduced budget per trial
PATIENCE     = 40         # early-stop patience during the sweep
SAMPLE_SEED  = 1          # seed for the random sampler (not the model seed)

# Fixed across every trial: this notebook is about the *nonlinear* model.
FIXED = dict(
    data_dir=DATA_DIR, val_fraction=VAL_FRACTION, seed=SEED, min_reads=MIN_READS,
    nonlinear=True, epochs=SWEEP_EPOCHS, early_stop_patience=PATIENCE,
    save_predictions=False, out_dir='/tmp/dki_sweep',
)


def iter_configs():
    keys = list(SPACE)
    if SEARCH == 'grid':
        for combo in itertools.product(*(SPACE[k] for k in keys)):
            d = dict(zip(keys, combo))
            if d['loss'] != 'composite':
                d['alpha'] = SPACE['alpha'][0]   # collapse the irrelevant axis
            yield d
    elif SEARCH == 'random':
        rng = random.Random(SAMPLE_SEED)
        seen = set()
        tries = 0
        while len(seen) < N_TRIALS and tries < N_TRIALS * 50:
            tries += 1
            d = {k: rng.choice(v) for k, v in SPACE.items()}
            if d['loss'] != 'composite':
                d['alpha'] = SPACE['alpha'][0]
            key = tuple(sorted(d.items()))
            if key in seen:
                continue
            seen.add(key)
            yield d
    else:
        raise ValueError(f"SEARCH must be 'random' or 'grid', got {SEARCH!r}")


configs = list(iter_configs())
# de-dupe (alpha collapse can create repeats) and report the budget
_uniq, _seen = [], set()
for d in configs:
    k = tuple(sorted(d.items()))
    if k not in _seen:
        _seen.add(k); _uniq.append(d)
configs = _uniq
print(f'{SEARCH} search: {len(configs)} trial(s), '
      f'{SWEEP_EPOCHS} epochs each (early stop @ {PATIENCE}).')

## 4. Run the sweep

Each trial reuses the shared `data` object, so the (potentially expensive) load
+ normalisation happens once. We record the val BC, held-out test BC (when
ground truth exists), parameter count, wall-clock, the best epoch, and how many
optimiser steps were skipped on non-finite gradients - a high count means the
config diverged and its val BC should be distrusted.

In [ ]:
rows = []
t_start = time.perf_counter()

for i, hp in enumerate(configs):
    bs = min(hp['batch_size'], N_TRAIN)   # cap to available training samples
    cfg = TrainConfig(
        **FIXED,
        lr=hp['lr'], min_lr=hp['min_lr'], batch_size=bs,
        hidden_mult=hp['hidden_mult'], weight_decay=hp['weight_decay'],
        grad_clip=hp['grad_clip'], loss=hp['loss'], alpha=hp['alpha'],
        consistency_weight=hp['consistency_weight'],
    )
    t0 = time.perf_counter()
    model, result, _ = train(cfg, data=data)
    wall = time.perf_counter() - t0

    test_bc = np.nan
    if HAS_TEST:
        qtst = predict(model, data.z_test, t_final=cfg.t_final)
        test_bc = bray_curtis(qtst.cpu(), data.p_test.cpu()).item()

    rows.append(dict(
        trial=i, **hp, batch_size_eff=bs,
        val_bc=result.best_val_loss, test_bc=test_bc,
        best_epoch=result.best_epoch, skipped=result.skipped_steps,
        n_params=sum(p.numel() for p in model.parameters()),
        wall_s=round(wall, 1),
    ))
    print(f'[{i+1:>3}/{len(configs)}] val_bc={result.best_val_loss:.4f} '
          f"test_bc={test_bc:.4f} skipped={result.skipped_steps:>3} "
          f"lr={hp['lr']:.0e} min_lr={hp['min_lr']:.0e} bs={bs} "
          f"hm={hp['hidden_mult']} wd={hp['weight_decay']:.0e} "
          f"gc={hp['grad_clip']} loss={hp['loss']} cw={hp['consistency_weight']} "
          f'({wall:.0f}s)')

results = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
results.to_csv('results/sweep_results.csv', index=False)
print(f'\nDone in {(time.perf_counter() - t_start) / 60:.1f} min - '
      f'wrote results/sweep_results.csv ({len(results)} rows).')

## 5. Leaderboard

Sort by val BC (the selection metric). `clean=True` rows had **no** skipped
steps - those are the trustworthy configs. A config with a great val BC but
many skipped steps got there by luck on the surviving batches and should be
treated with suspicion.

In [ ]:
results['clean'] = results['skipped'] == 0
leaderboard = results.sort_values('val_bc').reset_index(drop=True)

show_cols = ['val_bc', 'test_bc', 'clean', 'skipped', 'lr', 'min_lr',
             'batch_size_eff', 'hidden_mult', 'weight_decay', 'grad_clip',
             'loss', 'alpha', 'consistency_weight', 'best_epoch', 'n_params', 'wall_s']
print('Baseline (main notebook): lr=1e-3, min_lr=1e-5, bs=min(500,N), hidden_mult=2\n')
leaderboard[show_cols].head(15)

## 6. Which dimensions actually matter?

Per-dimension marginals: for each hyperparameter, how does val BC distribute
across the values it took? A dimension whose boxes are clearly separated is one
the model is sensitive to; overlapping boxes mean it barely matters in this
range. Diverged trials (`clean=False`) are drawn as red x so a value that
*tends* to blow up is visible even if its surviving runs look fine.

In [ ]:
dims = ['lr', 'min_lr', 'batch_size_eff', 'hidden_mult', 'weight_decay',
        'grad_clip', 'loss', 'consistency_weight']
ncol = 4
nrow = int(np.ceil(len(dims) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow))
axes = axes.flatten()

for ax, dim in zip(axes, dims):
    vals = sorted(results[dim].unique(), key=lambda x: (isinstance(x, str), x))
    data_clean = [results[(results[dim] == v) & results['clean']]['val_bc'].dropna() for v in vals]
    positions = np.arange(len(vals))
    ax.boxplot([d if len(d) else [np.nan] for d in data_clean],
               positions=positions, widths=0.5, showfliers=False)
    # overlay every trial as a point; red x for diverged
    for j, v in enumerate(vals):
        sub = results[results[dim] == v]
        ok = sub[sub['clean']]
        bad = sub[~sub['clean']]
        ax.scatter(np.full(len(ok), j) + np.random.uniform(-0.12, 0.12, len(ok)),
                   ok['val_bc'], s=14, alpha=0.6, color='#1f77b4')
        ax.scatter(np.full(len(bad), j), bad['val_bc'], s=40, marker='x', color='#d62728')
    ax.set_xticks(positions)
    ax.set_xticklabels([f'{v:g}' if not isinstance(v, str) else v for v in vals],
                       rotation=30, fontsize=8)
    ax.set_title(dim, fontsize=10); ax.set_ylabel('val BC'); ax.grid(alpha=0.3, axis='y')

for ax in axes[len(dims):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 7. The lr x batch_size interaction

These two are the classic coupled pair (effective step size scales with both),
so a marginal view can be misleading. Heatmap of the **best clean val BC** in
each `lr` x `batch_size` cell; blank cells were never sampled (expected in
random search).

In [ ]:
piv = (results[results['clean']]
       .pivot_table(index='lr', columns='batch_size_eff', values='val_bc', aggfunc='min')
       .sort_index(ascending=False))
fig, ax = plt.subplots(figsize=(1.3 * piv.shape[1] + 2, 1.0 * piv.shape[0] + 1.5))
im = ax.imshow(piv.values, cmap='viridis_r', aspect='auto')
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels([f'{v:.0e}' for v in piv.index])
ax.set_xlabel('batch_size'); ax.set_ylabel('lr'); ax.set_title('best clean val BC')
for r in range(piv.shape[0]):
    for c in range(piv.shape[1]):
        v = piv.values[r, c]
        if not np.isnan(v):
            ax.text(c, r, f'{v:.3f}', ha='center', va='center',
                    color='w' if v > np.nanmedian(piv.values) else 'k', fontsize=8)
fig.colorbar(im, ax=ax, label='val BC'); plt.tight_layout(); plt.show()

## 8. (Optional) Re-check the top configs across seeds

A single seed sets both the weight init and the minibatch order, so the #1
config might just be lucky. Re-train the top `TOP_K` clean configs over a few
seeds and rank by **mean** val BC - a config that wins on average is the one to
ship. Set `RECHECK=False` to skip.

In [ ]:
RECHECK = True
TOP_K   = 3
RECHECK_SEEDS = [0, 1, 2]

best_hp = None
if RECHECK and len(leaderboard):
    top = leaderboard[leaderboard['clean']].head(TOP_K)
    hp_keys = ['lr', 'min_lr', 'batch_size', 'hidden_mult', 'weight_decay',
               'grad_clip', 'loss', 'alpha', 'consistency_weight']
    rc_rows = []
    for _, row in top.iterrows():
        hp = {k: row[k] for k in hp_keys}
        hp['batch_size'] = int(row['batch_size_eff'])
        for s in RECHECK_SEEDS:
            d = load_dataset(DATA_DIR, val_fraction=VAL_FRACTION, seed=s, min_reads=MIN_READS)
            cfg = TrainConfig(
                **{**FIXED, 'seed': s}, lr=hp['lr'], min_lr=hp['min_lr'],
                batch_size=min(hp['batch_size'], d.z_train.shape[0]),
                hidden_mult=int(hp['hidden_mult']), weight_decay=hp['weight_decay'],
                grad_clip=hp['grad_clip'], loss=hp['loss'], alpha=hp['alpha'],
                consistency_weight=hp['consistency_weight'],
            )
            _, res, _ = train(cfg, data=d)
            rc_rows.append(dict(orig_trial=int(row['trial']), seed=s,
                                val_bc=res.best_val_loss, skipped=res.skipped_steps))
            print(f"trial {int(row['trial'])} seed {s}: val_bc={res.best_val_loss:.4f}")
    recheck = pd.DataFrame(rc_rows)
    agg = (recheck.groupby('orig_trial')['val_bc']
           .agg(['mean', 'std', 'max']).sort_values('mean'))
    display(agg)
    winner = int(agg.index[0])
    best_hp = {k: leaderboard.loc[leaderboard['trial'] == winner, k].iloc[0] for k in hp_keys}
    best_hp['batch_size'] = int(leaderboard.loc[leaderboard['trial'] == winner, 'batch_size_eff'].iloc[0])
    print('\nMost robust config (lowest mean val BC across seeds): trial', winner)
else:
    print('Recheck skipped - taking the single-seed leaderboard winner.')

## 9. Retrain the winner at the full budget and save

Take the chosen config (seed-robust winner if section 8 ran, else the
single-seed #1), retrain at the **full** epoch budget with prediction-saving
on, and write the checkpoint to `results/best_model.pt` plus the resolved
config to `results/best_config.json`. Point the keystoneness workflow's
`TrainConfig` at these values to compute keystoneness on the best model.

In [ ]:
FINAL_EPOCHS   = 400
FINAL_PATIENCE = 50

hp_keys = ['lr', 'min_lr', 'batch_size', 'hidden_mult', 'weight_decay',
           'grad_clip', 'loss', 'alpha', 'consistency_weight']
if best_hp is None:
    win = leaderboard.iloc[0]
    best_hp = {k: win[k] for k in hp_keys}
    best_hp['batch_size'] = int(win['batch_size_eff'])

best_cfg = TrainConfig(
    data_dir=DATA_DIR, out_dir='results', val_fraction=VAL_FRACTION, seed=SEED,
    min_reads=MIN_READS, nonlinear=True, save_predictions=True,
    epochs=FINAL_EPOCHS, early_stop_patience=FINAL_PATIENCE,
    lr=best_hp['lr'], min_lr=best_hp['min_lr'],
    batch_size=min(int(best_hp['batch_size']), N_TRAIN),
    hidden_mult=int(best_hp['hidden_mult']), weight_decay=best_hp['weight_decay'],
    grad_clip=best_hp['grad_clip'], loss=best_hp['loss'], alpha=best_hp['alpha'],
    consistency_weight=best_hp['consistency_weight'],
)
print('Final config:')
for k in hp_keys:
    print(f'  {k:>18} = {getattr(best_cfg, k)}')

model, result, _ = train(best_cfg, data=data)
with open('results/best_config.json', 'w') as f:
    json.dump(best_cfg.__dict__, f, indent=2)

test_bc = np.nan
if HAS_TEST:
    test_bc = bray_curtis(predict(model, data.z_test).cpu(), data.p_test.cpu()).item()
print(f'\nBest val BC: {result.best_val_loss:.4f} @ epoch {result.best_epoch}'
      + (f'   held-out test BC: {test_bc:.4f}' if HAS_TEST else ''))
print('Saved results/best_model.pt and results/best_config.json')

plt.figure(figsize=(7, 4))
plt.plot(result.train_loss, label='train loss', alpha=0.7)
plt.plot(result.val_loss, label='val BC', alpha=0.9)
plt.axvline(result.best_epoch, ls='--', c='k', lw=1, label=f'best @ {result.best_epoch}')
plt.xlabel('epoch'); plt.ylabel('loss / BC'); plt.legend()
plt.title('Winner - full-budget retrain'); plt.show()

## 10. Hand off to keystoneness

`results/best_config.json` holds the winning `TrainConfig`. To compute
keystoneness on this model, open `dki_keystoneness_workflow.ipynb` and replace
the hand-picked `cfg = TrainConfig(...)` in its section 3 with these values (or
load the JSON):

```python
import json
from dki.train import TrainConfig
cfg = TrainConfig(**json.load(open('results/best_config.json')))
```

Then run its sections 4-9 (predict -> removal experiment ->
structural/functional keystoneness) unchanged. Because keystoneness is a
counterfactual *derived from the trained model*, a lower val BC here means more
trustworthy keystone rankings downstream.

In [ ]:
# Convenience: download the sweep artefacts when on Colab.
print('Artefacts in results/:')
for f in ['sweep_results.csv', 'best_config.json', 'best_model.pt']:
    p = os.path.join('results', f)
    print(' ', p, 'ok' if os.path.exists(p) else '(missing)')
try:
    from google.colab import files
    for f in ['sweep_results.csv', 'best_config.json', 'best_model.pt']:
        files.download(os.path.join('results', f))
except Exception as e:
    print('Not in Colab or download skipped:', e)